# 1.2 Attention 机制基础

本节回答两个问题：**Attention 在算什么**（用比喻和图建立直觉），以及**它的标准计算流程长什么样**（为 1.3 节分析性能瓶颈做准备）。

如果你已经熟悉 Attention，可以快速浏览本节，重点确认「三步计算」的记号约定（Q、K、V、S、P、O）——后面所有章节都会沿用这套记号。

## 一、为什么需要 Attention：词的意思取决于上下文

看这句话：**“苹果发布了新手机”**。

“苹果”这个词的向量应该表达“公司”还是“水果”？单独看这个词无法回答——**必须看它周围的词**。语言模型的核心任务之一，就是让每个 token 的表示吸收上下文信息。

Attention 给出的方案非常直白，像**查字典**：

1. 每个词提出一个“查询”（我要找什么信息）；
2. 每个词同时亮出一个“标签”（我能提供什么信息）；
3. 查询和标签两两比对打分——分数越高，说明这个词对我的参考价值越大；
4. 按分数把所有词的内容加权混合，作为我更新后的表示。

这就是 Attention 的全部直觉。剩下的只是把它写成矩阵运算。

## 二、Q、K、V 的由来

![Q、K、V 的由来](images/attention_qkv.svg)

同一份输入 `X`（n 个 token、每个 d 维），分别乘三组**可学习的权重矩阵**：

- `Q = X · W_Q`：**查询**（Query）——当前 token 想找什么；
- `K = X · W_K`：**键**（Key）——每个 token 亮出的“标签”，用来被别人匹配；
- `V = X · W_V`：**值**（Value）——每个 token 真正提供的内容，被加权取走。

> 记号约定（本章沿用）：`n` = 序列长度（token 个数），`d` = 单个头的向量维度（Head-Dim）。Q/K/V 形状均为 `n × d`。

In [ ]:
import numpy as np
np.random.seed(0)

n, d_model, d = 4, 16, 8   # 4 个 token；模型维 16；单头维 8

X = np.random.randn(n, d_model).astype(np.float32)          # 输入 (n, d_model)
W_Q = (np.random.randn(d_model, d) * 0.3).astype(np.float32)
W_K = (np.random.randn(d_model, d) * 0.3).astype(np.float32)
W_V = (np.random.randn(d_model, d) * 0.3).astype(np.float32)

Q = X @ W_Q    # (n, d) 查询
K = X @ W_K    # (n, d) 键
V = X @ W_V    # (n, d) 值
print('X:', X.shape, ' Q:', Q.shape, ' K:', K.shape, ' V:', V.shape)

## 三、Attention 的三步计算

![Attention 三步计算](images/attention_three_steps.svg)

整体公式（先记住它，下面逐步拆开）：

$$\text{Attention}(Q, K, V) = \text{Softmax}\!\left(\frac{Q K^{T}}{\sqrt{d}}\right) V$$

### 第①步 打分：$S = Q K^{T} / \sqrt{d}$

`Q · Kᵀ` 的第 `(i, j)` 元素是第 `i` 个查询与第 `j` 个键的**点积**。点积衡量两个向量的方向一致性——越“对口”，分数越高。得到 `n × n` 的得分矩阵 `S`。

### 第②步 归一化：$P = \text{Softmax}(S)$（按行）

对 `S` 的**每一行**做 Softmax，把任意范围的分数变成一行“和为 1”的非负权重 `P`。第 `i` 行描述：第 `i` 个 token 把注意力按什么比例分给序列中的每个位置。

### 第③步 加权求和：$O = P · V$

用权重 `P` 对所有 `V` 加权求和。输出的第 `i` 行就是第 `i` 个 token 吸收上下文后的新表示。

In [ ]:
# 三步计算：每一步都打印形状，帮助建立“矩阵流动”的感觉
S = Q @ K.T / np.sqrt(d)                       # ① 打分  (n, n)
P = np.exp(S - S.max(axis=1, keepdims=True))   # ② 归一化之 exp（减最大值防溢出，细节见 1.4 节）
P = P / P.sum(axis=1, keepdims=True)
O = P @ V                                       # ③ 加权求和 (n, d)

print('S:', S.shape, ' P:', P.shape, ' O:', O.shape)
print('P 每行的和（应全为 1）：', P.sum(axis=1))

## 四、Softmax 的直觉：放大版的“赢家通吃”

![Softmax 直觉](images/softmax_intuition.svg)

把注意力权重中的一行单独拿出来看。运行下面的代码，观察：得分最高的位置拿到接近 1 的权重，其他位置几乎归零——Softmax 会**把分数的差距指数级放大**。

$$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

In [ ]:
i = 0   # 看第 0 个 token 的注意力分配
print('第 %d 行得分 S[i]：' % i, np.round(S[i], 3))
print('第 %d 行权重 P[i]：' % i, np.round(P[i], 3))
print('权重之和：', P[i].sum())
print('\n文本条形图（* 越多权重越大）：')
for j in range(n):
    bar = '*' * int(round(P[i, j] * 40))
    print(f'  token {j}: {P[i,j]:.3f} {bar}')

## 五、为什么除以 $\sqrt{d}$？

直觉：两个随机 d 维向量的点积，幅度大约是 $\sqrt{d}$ 量级。`d` 越大，分数的**方差**越大；分数差距一旦过大，Softmax 就会饱和成“非零即一”的独热分布——注意力和梯度都失去了区分度。

除以 $\sqrt{d}$ 恰好把分数方差拉回约 1 的量级，让 Softmax 工作在“平滑可学习”的区间。用实验验证：

In [ ]:
# 观察：随机向量的点积幅度随维度 d 的增长
for d_test in [8, 64, 512, 4096]:
    q = np.random.randn(d_test)
    k = np.random.randn(d_test)
    print(f'd = {d_test:5d} 时，随机点积 q·k ≈ {q @ k:+8.2f}   （√d = {np.sqrt(d_test):6.2f}）')

## 六、多头注意力：多组“视角”并行

一个头只有 d 维（如 128），只够表达一种“关注模式”。Transformer 让多组 QKV 并行计算 Attention，再把各头的输出拼接、投影，融合多个视角（语法头、指代头、语义头……）。

![多头注意力](images/multi_head_attention.svg)

关键点：**每个头独立做一遍完整的三步计算**——所以“单个 Attention 怎么算得快”的问题，乘以头数后被原样放大。

In [13]:
def attention(Q, K, V):
    """标准 Attention，本章反复使用这个参考实现。"""
    S = Q @ K.T / np.sqrt(Q.shape[-1])
    P = np.exp(S - S.max(axis=-1, keepdims=True))
    P = P / P.sum(axis=-1, keepdims=True)
    return P @ V

h, d_head = 4, 8                    # 4 个头，每头 8 维
outputs = []
for head in range(h):
    Wq = (np.random.randn(d, d_head) * 0.3).astype(np.float32)   # 每个头自己的投影
    Wk = (np.random.randn(d, d_head) * 0.3).astype(np.float32)
    Wv = (np.random.randn(d, d_head) * 0.3).astype(np.float32)
    outputs.append(attention(Q @ Wq, K @ Wk, V @ Wv))            # 各头独立计算

O_concat = np.concatenate(outputs, axis=-1)   # 拼接 (n, h*d_head)
print('每个头的输出:', outputs[0].shape, ' 拼接后:', O_concat.shape)

每个头的输出: (4, 8)  拼接后: (4, 32)


## 七、GQA / MQA 与 KV Cache：推理时代的主角

生成式推理（逐字生成）有两个标配优化，它们也是 `FusedInferAttentionScore` 算子接口中大量参数存在的原因：

### KV Cache：历史只算一次

逐字生成时，第 `t` 步只多出一个新 token。历史 token 的 K、V **不会变**——算一次存进缓存（KV Cache），之后每步直接读。Attention 的计算变成「新 token 的 1 行 Q × 缓存里全部 K/V」。

![KV Cache](images/kv_cache.svg)

### GQA / MQA：给 KV Cache 瘦身

多头注意力要为每个头缓存独立的 K、V，显存占用巨大。GQA/MQA 让**多个 Q 头共享同一组 KV**：

| 方案 | Q 头数 | KV 组数 | 效果 |
|--|--|--|--|
| MHA（标准多头） | h | h | 无节省 |
| GQA（分组查询） | h | g（如 h/8） | 缓存缩到 g/h，质量损失极小（Llama 2/3 采用） |
| MQA（多查询） | h | 1 | 缓存缩到 1/h，质量损失较明显 |

GQA 带来的计算变化：共享同一组 K/V 的那些 Q 头可以**拼成一个大 Q 矩阵一次性计算**——这正是 FlashAttention 类算子（含 `FusedInferAttentionScore`）在推理场景的典型工作负载。

In [12]:
# GQA 最小示例：2 个 Q 头共享 1 组 KV
h_q, h_kv, d_head = 2, 1, 8

Q_gqa = np.random.randn(h_q, n, d_head).astype(np.float32)   # 2 个 Q 头
K_gqa = np.random.randn(h_kv, n, d_head).astype(np.float32)  # 1 组 KV（共享）
V_gqa = np.random.randn(h_kv, n, d_head).astype(np.float32)

outs = [attention(Q_gqa[q], K_gqa[0], V_gqa[0]) for q in range(h_q)]
print('2 个 Q 头各自输出:', outs[0].shape)
print('实现上常把共享 Q 拼成 (h_q*n, d_head) 一次算完 —— 算子里的 num_heads/num_kv_heads 参数即对应此处')

2 个 Q 头各自输出: (4, 8)
实现上常把共享 Q 拼成 (h_q*n, d_head) 一次算完 —— 算子里的 num_heads/num_kv_heads 参数即对应此处


## 八、本节小结

1. Attention = **打分 → 归一化 → 加权求和**：`S = QKᵀ/√d`，`P = Softmax(S)`，`O = PV`。
2. Softmax 按**行**归一化——这个“必须凑齐整行”的性质，正是 1.4 节 Online Softmax 要攻克的对象。
3. 除以 `√d` 是为了控制分数方差，避免 Softmax 饱和。
4. 多头 = 多组视角并行；GQA/MQA 让多个 Q 头共享 KV；KV Cache 让历史 K/V 只算一次。
5. 观察三步计算：`S` 和 `P` 都是 **n × n** 矩阵——序列一长它们就变得巨大。它们去哪儿了？这是下一节的伏笔。

## 章节测验

1. Q、K、V 三者中，谁的形状是 `n × d`？三者都来自哪里？
2. `S = QKᵀ/√d` 中，`S[i, j]` 的含义是什么？为什么 S 是 `n × n` 而不是 `n × d`？
3. Softmax 是按行还是按列做的？为什么必须是这个方向？
4. 不除以 `√d` 会有什么问题？请用本节实验中的现象回答。
5. GQA 相比 MHA 节省的是什么资源？为什么推理阶段尤其重要？
6. （思考题）KV Cache 中存的是 Q、K、V 中的哪几个？为什么没有剩下的那个？

> 答案见 `answer/01.02_answer.txt`，建议先自己作答再核对。

下一节：[1.3 标准 Attention 的访存瓶颈](01.03_attention_bottleneck.ipynb)——这套三步计算放到真实芯片上，慢在哪里？